# Colab setup
Run the cells in order. The clone cell can be rerun: it updates an existing checkout and checks that the GitHub repository contains the project files before installation.

## 1. Check GPU availability

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Clone or update the GitHub repository

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/tiendinhquang2104/multimodal-churn-fusion.git'
PROJECT_DIR = Path('/content/multimodal-churn-fusion')

if (PROJECT_DIR / '.git').is_dir():
    head = subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'rev-parse', '--verify', 'HEAD'],
        capture_output=True,
    )
    if head.returncode == 0:
        subprocess.run(
            ['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', 'main'],
            check=True,
        )
    else:
        # Recover the empty checkout made before the first project commit.
        subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', '-B', 'main', 'origin/main'], check=True)
elif PROJECT_DIR.exists():
    raise RuntimeError(f'{PROJECT_DIR} exists but is not a Git checkout. Inspect it before cloning.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)

required = ['pyproject.toml', 'requirements.txt', 'configs/baseline.yaml', 'src/multimodal_churn']
missing = [name for name in required if not (PROJECT_DIR / name).exists()]
if missing:
    raise RuntimeError(
        'The GitHub checkout is missing project files: ' + ', '.join(missing)
        + '. Push the local project to GitHub, then rerun this cell.'
    )
print('Repository ready:', PROJECT_DIR)

## 3. Move into the verified project

In [ ]:
%cd /content/multimodal-churn-fusion

## 4. Install runtime requirements

In [ ]:
%pip install -r requirements.txt

## 5. Install the package in editable mode

In [ ]:
%pip install -e .

## 6. Mount Google Drive
An "already mounted" message is normal when rerunning this cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 7. Configure dataset and output paths
Set the two root paths for your Drive mount. The active dataset name comes from baseline.yaml; change that YAML value and add a matching dataset config for a future benchmark. The raw dataset folder is never created or modified here.

In [ ]:
from pathlib import Path
from multimodal_churn.utils.config import load_config, resolve_dataset_paths

baseline_path = Path('configs/baseline.yaml')
DATASET_NAME = load_config(baseline_path)['dataset']['name']
DATASETS_ROOT = Path('/content/drive/MyDrive/Project Master Thesis/Data/Datasets')
ARTIFACTS_ROOT = Path('/content/drive/MyDrive/multimodal-churn-fusion')

dataset_config = Path('configs/datasets') / f'{DATASET_NAME}.yaml'
config = load_config(baseline_path, dataset_path=dataset_config)
dataset_paths = resolve_dataset_paths(config, DATASETS_ROOT)
DATASET_PATH = dataset_paths['folder']
DATASET_ARCHIVE = dataset_paths.get('archive')
EMBEDDINGS_PATH = ARTIFACTS_ROOT / 'data' / 'embeddings' / DATASET_NAME
CHECKPOINTS_PATH = ARTIFACTS_ROOT / 'outputs' / 'checkpoints' / DATASET_NAME
for path in (EMBEDDINGS_PATH, CHECKPOINTS_PATH):
    path.mkdir(parents=True, exist_ok=True)

print('Dataset folder:', DATASET_PATH)
print('Dataset archive:', DATASET_ARCHIVE)
print('Archive exists:', DATASET_ARCHIVE.is_file() if DATASET_ARCHIVE else 'not configured')
print('Embeddings:', EMBEDDINGS_PATH)
print('Checkpoints:', CHECKPOINTS_PATH)

## 8. Verify package imports and configuration

In [ ]:
from multimodal_churn import MultimodalChurnModel
from multimodal_churn.models.encoders import TransactionEncoder, TextEncoder, ImageEncoder
from multimodal_churn.models.fusion import ConcatFusion
print('Package import OK; dataset:', config['dataset']['name'])
print('Embedding dimension:', config['model']['common_embedding_dim'])